In [ ]:
# 1. Install system dependencies for PDF processing
!apt-get install -y poppler-utils
!pip install bitsandbytes accelerate chandra-ocr[hf]
# 2. Install Chandra OCR
!pip install chandra-ocr[hf]


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (250 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from chandra.model import InferenceManager
from chandra.model.schema import BatchInputItem
from PIL import Image
import torch
import gc
from transformers import BitsAndBytesConfig, AutoModelForImageTextToText, AutoProcessor
from google.colab import userdata

# 1. Force clear GPU memory
if 'manager' in locals():
    del manager
gc.collect()
torch.cuda.empty_cache()

# 2. Get HF token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
    print('Warning: HF_TOKEN not found in Secrets. Ensure you have added it and enabled notebook access.')

# 3. Define 4-bit config to save memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

# 4. Initialize Manager with the correct model ID
model_id = 'datalab-to/chandra-ocr-2'

print(f'Loading {model_id} in 4-bit...')
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map='auto',
    trust_remote_code=True,
    token=hf_token
)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, token=hf_token)

# Manually assemble the manager
manager = InferenceManager(method='hf')
manager.model = model
manager.processor = processor
print('Model loaded successfully!')

Loading datalab-to/chandra-ocr-2 in 4-bit...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.6G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

ValueError: The current `device_map` had weights offloaded to the disk, which needed to be re-saved. This is either because the weights are not in `safetensors` format, or because the model uses an internal weight format different than the one saved (i.e. most MoE models). Please provide an `offload_folder` for them in `from_pretrained`.

In [ ]:
# 3. Process your worksheet
# Ensure the model has the processor attached as expected by the library
if not hasattr(manager.model, "processor"):
    manager.model.processor = manager.processor

img = Image.open("/content/Screenshot 2026-04-19 at 9.42.03 PM.png")
batch = [BatchInputItem(image=img, prompt_type="ocr_json")]

results = manager.generate(batch)
print(results[0].markdown)

In [ ]:
pip install markdown beautifulsoup4

In [ ]:
import json
from bs4 import BeautifulSoup

def extract_content(ocr_data):
    structured_results = []
    # Navigate the 'children' structure
    for page in ocr_data['children']:
        for block in page['children']:
            block_id = block['id']
            block_type = block['block_type']

            # Extract raw text from the HTML snippet
            soup = BeautifulSoup(block['html'], 'html.parser')
            text_content = soup.get_text().strip()

            structured_results.append({
                "id": block_id,
                "type": block_type,
                "content": text_content,
                "bbox": block['bbox']
            })
    return structured_results

# Example: The first block in your JSON is the 'To prove' statement (The Question)
# The subsequent blocks are the student's proof steps.

Q15. Briefly state the first two of the Four Noble Truths taught by Gautama Buddha.
2 Marks
The first two of the four noble truths are: 1) The truth of suffering (dukkha): Life is inherently marked by suffering, dissatisfaction, and pain. 2) The truth of the cause of suffering (samudaya): Suffering is caused by craving, desire, and attachment to things like Pleasure, Possessions, and existence.


In [ ]:
from sentence_transformers import SentenceTransformer, util

# Small and fast model for M1
model = SentenceTransformer('all-MiniLM-L6-v2')

def check_semantic_similarity(student_step, rubric_step):
    embeddings = model.encode([student_step, rubric_step])
    similarity = util.cos_sim(embeddings[0], embeddings[1])
    return similarity.item()

# Example usage
sim = check_semantic_similarity("Taking from LHS", "Starting with the left hand side")
print(f"Match: {sim:.2f}") # Output: ~0.90

In [ ]:
from sympy.parsing.latex import parse_latex
from sympy import simplify, symbols, sin, cos, sec, csc

def are_math_steps_equivalent(student_latex, teacher_latex):
    try:
        # Define symbols used in the worksheet
        theta = symbols('theta')

        # Parse LaTeX to SymPy objects
        expr_student = parse_latex(student_latex)
        expr_teacher = parse_latex(teacher_latex)

        # Check if they are mathematically identical
        # (Expression A - Expression B) should simplify to 0
        return simplify(expr_student - expr_teacher) == 0
    except Exception as e:
        return False

# Example from your JSON:
s_latex = r"1 + 1"
t_latex = r"2"
print(f"Math Match: {are_math_steps_equivalent(s_latex, t_latex)}") # True